# 22 — Direction 1: submission

The policy notebook 21 trained (`d1_policy.pt`) run by its own code (`d1_policy.py`, the repo's
`hvac_controller/d1/policy.py`: NumPy + torch on CPU, ≈ 4 ms per step) inside the organisers' inference server.

**Setup:** Internet **off**; CPU; attach the competition data and the **policy dataset** (notebook 21's
`/kaggle/working/data.zip`, uploaded as a Kaggle dataset: `d1_policy.pt`, `d1_policy.py`, `README.txt`). A normal Save
Version plays the public example (atlanta_2, ≈ 15 min); Submit runs the hidden scenarios.

In [ ]:
import sys
from pathlib import Path

INPUT = Path("/kaggle/input")
candidates = [INPUT / "competitions" / "neurips-2026-smart-buildings", INPUT / "neurips-2026-smart-buildings"]
candidates += [p.parent for p in sorted(INPUT.rglob("kaggle_evaluation")) if (p.parent / "smart_buildings").exists()]
DATA_ROOT = next(p for p in candidates if (p / "smart_buildings").exists() and (p / "kaggle_evaluation").exists())
sys.path.insert(0, str(DATA_ROOT))
sys.path.insert(0, str(DATA_ROOT / "kaggle_evaluation"))
SUB = next(p.parent for p in sorted(INPUT.rglob("d1_policy.py")) if (p.parent / "d1_policy.pt").exists())
sys.path.insert(0, str(SUB))
print("competition data:", DATA_ROOT, "| policy:", SUB, (SUB / "README.txt").read_text() if (SUB / "README.txt").exists() else "")

import torch
torch.set_num_threads(1)
from d1_policy import KagglePolicy
from kaggle_evaluation import smart_buildings_inference_server


class SubmissionPolicy(smart_buildings_inference_server.Policy):
    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        self.policy = KagglePolicy(building_meta_data, action_spec, weights=SUB / "d1_policy.pt")

    def get_action(self, obs):
        return self.policy.get_action(obs)

In [ ]:
smart_buildings_inference_server.SmartBuildingsInferenceServer(SubmissionPolicy).run(
    competition_data_folder=str(DATA_ROOT),
)